# NLA — Step 2: Teacher summaries

A stronger open model (Qwen2.5-3B-Instruct, 4-bit) reads each text snippet and
describes what the target model is 'thinking about' there. These summaries are the
warm-start targets for the verbalizer and reconstructor.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

This step is resumable: if Colab disconnects, just re-run the generate cell and it
continues where it left off.

In [ ]:
!nvidia-smi

## 1. Get the (latest) project code

In [ ]:
REPO_URL = "https://github.com/mohamedibrahim26/nla-kth.git"

import os
repo_dir = "/content/" + REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if os.path.exists(repo_dir):
    !cd $repo_dir && git pull
else:
    !git clone $REPO_URL
%cd $repo_dir
!pip install -q -r requirements.txt

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/nla/data'
print('Data dir:', DATA_DIR)
assert os.path.exists(os.path.join(DATA_DIR, 'metadata.jsonl')), 'Run Step 1 (harvest) first!'

## 3. Smoke test (50 summaries)

Confirm quality looks reasonable before committing to the full run.

In [ ]:
!python src/generate_teacher_summaries.py --data_dir "$DATA_DIR" --batch_size 16 --limit 50

## 4. Inspect a few (snippet -> teacher summary)

In [ ]:
import json
meta = {json.loads(l)['idx']: json.loads(l)['text'] for l in open(os.path.join(DATA_DIR,'metadata.jsonl'))}
sums = [json.loads(l) for l in open(os.path.join(DATA_DIR,'summaries.jsonl'))]
print('total summaries so far:', len(sums))
for row in sums[:3]:
    print('\n' + '='*80)
    print('SNIPPET :', meta[row['idx']][:250])
    print('\nTEACHER :', row['summary'])

## 5. Full run

When the smoke test looks good, run this to summarise all snippets. It will skip the
50 already done and continue. Re-run it if Colab disconnects.

In [ ]:
!python src/generate_teacher_summaries.py --data_dir "$DATA_DIR" --batch_size 16